# 01_setup: Configuração e Extração para Landing

Este notebook realiza a criação da estrutura de dados no Databricks (Schemas e Volumes) e extrai os dados do MongoDB Atlas para a Landing Zone no formato JSON.

## 1. Criando Volumes e Schemas
Executamos os comandos SQL para garantir que a arquitetura Medalhão esteja preparada.

In [3]:
%sql
CREATE CATALOG IF NOT EXISTS workspace;

CREATE SCHEMA IF NOT EXISTS workspace.landing
COMMENT 'Schema/Database para dados brutos';

CREATE VOLUME IF NOT EXISTS workspace.landing.dados
COMMENT 'Volume para dados brutos criados no schema/database landing';

CREATE SCHEMA IF NOT EXISTS workspace.bronze
COMMENT 'Schema/Database para dados bronze (delta)';

CREATE SCHEMA IF NOT EXISTS workspace.silver
COMMENT 'Schema/Database para dados silver (delta)';

CREATE SCHEMA IF NOT EXISTS workspace.gold
COMMENT 'Schema/Database para dados gold (delta) - modelagem dimensional';

SyntaxError: invalid syntax (1590299965.py, line 2)

## 2. Conexão com o MongoDB Atlas
Certifique-se de que as variáveis de ambiente (DB_USER, DB_PASSWORD, DB_HOST) estão configuradas no seu cluster Databricks.

In [1]:
import os
import json
from pymongo import MongoClient

# Coletando as credenciais das variáveis de ambiente
DB_USER     = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST     = os.getenv('DB_HOST')
DB_NAME     = 'VendasArCondicionado'

# Montando a URI do Atlas
uri = f'mongodb+srv://{DB_USER}:{DB_PASSWORD}@{DB_HOST}/?retryWrites=true&w=majority'

try:
    # Instanciando o client
    client = MongoClient(uri)
    db = client[DB_NAME]
    
    # Teste de conexão
    client.admin.command('ping')
    
    print(f'✅ Conectado ao MongoDB Atlas com sucesso!')
    print(f'📂 Database ativa: {DB_NAME}')
    
except Exception as e:
    print(f'❌ Erro ao conectar: {e}')

✅ Conectado ao MongoDB Atlas com sucesso!
📂 Database ativa: VendasArCondicionado


## 3. Extração das Coleções para o Volume (Landing Zone)
Iteramos por todas as coleções, convertemos para JSON e salvamos no Volume `/Volumes/workspace/landing/dados/`.

In [4]:
colecoes = db.list_collection_names()
colecoes.sort()

print(f'{len(colecoes)} coleções encontradas:')

# Caminho do Volume no Databricks
caminho_landing = '/Volumes/workspace/landing/dados'

for colecao_nome in colecoes:
    # Buscar documentos
    cursor = db[colecao_nome].find()
    dados = list(cursor)
    
    if not dados:
        print(f'  ⚠️ {colecao_nome} está vazia. Pulando...')
        continue

    # Converter para JSON (tratando ObjectIds como string)
    json_str = json.dumps(dados, default=str, ensure_ascii=False)
    
    # Definir o caminho de destino no Volume
    caminho_arquivo = f'{caminho_landing}/{colecao_nome}.json'
    
    # SALVANDO NO DATABRICKS VOLUME
    dbutils.fs.put(caminho_arquivo, json_str, overwrite=True)
    
    print(f'  ✅ Coleção "{colecao_nome}" ({len(dados)} documentos) salva em: {caminho_arquivo}')

print('\n🚀 Extração para Landing finalizada!')

4 coleções encontradas:


NameError: name 'dbutils' is not defined